## **Search Evaluation**

Now that we have ground truth data, we can evaluate how well our search retrieves the correct documents.

For each question in our ground truth dataset, we run search. Then we check whether the results include the correct document.

## Setting up search

Create a new notebook for search evaluation. We'll use the ground truth CSV from the previous lesson and set up the search index here.

For search evaluation, we only need the search part of the RAG pipeline. We don't need to call the LLM yet.

Load the ground truth file from the previous notebook:

In [1]:
import pandas as pd

df_ground_truth = pd.read_csv("data/ground_truth-new.csv")
ground_truth = df_ground_truth.to_dict(orient="records")

Use the same *ingest.py* file we downloaded in the previous notebook.

Load the documents and build a minsearch index:

In [2]:
from ingest import load_faq_data, build_index

documents = load_faq_data()

documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

documents = documents_llm
index = build_index(documents)

Wrap the search call in a function called *text_search*. The name is deliberate. Later we'll write *vector_search* or a hybrid version and run the exact same evaluation on it. Everything downstream only needs a function that takes a query and returns results, so we can swap one for another. That mirrors how RAG works: the retrieval step doesn't care which search function sits behind it.

In [3]:
def text_search(query):
    boost_dict = {"question": 3.0, "section": 0.5}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict
    )

## Collecting relevance data

Start with one ground truth record:

In [4]:
q = ground_truth[0]
q

{'question': 'Is it okay to join the course late if I just found it now?',
 'document': '74eb249bbf'}

Run search for this question:

In [5]:
doc_id = q["document"]
results = text_search(query=q["question"])

First, compare the retrieved document IDs with the correct document ID:

In [6]:
for d in results:
    print(f'{d["id"]} == {doc_id}: {d["id"] == doc_id}')

74eb249bbf == 74eb249bbf: True
0fab61eca2 == 74eb249bbf: False
610ccb23c0 == 74eb249bbf: False
977bf7786c == 74eb249bbf: False
acf8fa5356 == 74eb249bbf: False


Then turn this comparison into a relevance list. In this lesson, relevance means whether a retrieved document is the correct document for this question.

In [7]:
relevance = []

for d in results:
    relevance.append(int(d["id"] == doc_id))

relevance

[1, 0, 0, 0, 0]

This gives a list of *0* and *1* values. *1* means the retrieved document has the same ID as the correct document.

Put this logic into a function:

In [8]:
def compute_relevance_text(q):
    doc_id = q["document"]
    results = text_search(query=q["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["id"] == doc_id))

    return relevance

For the first ground truth record, the relevance list is:

In [9]:
q = ground_truth[0]
print(q["question"])
compute_relevance_text(q)
# [1, 0, 0, 0, 0]

Is it okay to join the course late if I just found it now?


[1, 0, 0, 0, 0]

The correct document was the first search result.

Here are two more examples from the generated ground truth data.

For this question:

In [10]:
q = ground_truth[11]
print(q["question"])
compute_relevance_text(q)
# [1, 0, 0, 0, 0]

Where can I find the live stream link for the course office hours before it starts?


[1, 0, 0, 0, 0]

Here, the correct document was also the first search result.

For this question:

In [11]:
from tqdm.auto import tqdm

def compute_relevance_total_text(ground_truth):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance_text(q)
        relevance_total.append(relevance)

    return relevance_total

Call it for the first 15 ground truth questions:

In [12]:
ground_truth_sample = ground_truth[:15]
relevance_total_text = compute_relevance_total_text(ground_truth_sample)

  0%|          | 0/15 [00:00<?, ?it/s]

Look at the results:

In [13]:
relevance_total_text

[[1, 0, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 1, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [0, 0, 0, 1, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0]]

For the data we prepared on May 29, 2026, this gives:

In [ ]:
[
    [1, 0, 0, 0, 0],
    [0, 1, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [0, 0, 0, 0, 0],
    [0, 1, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [0, 0, 1, 0, 0],
    [1, 0, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [1, 0, 0, 0, 0],
]

Each entry in *relevance_total_text* is a relevance list. This is enough to check that the function works before we run it for the full dataset.

Next, make the relevance functions generic. We start with text search, but later we may want to evaluate vector search, hybrid search, or another retrieval method. The relevance logic is the same. Only the search function changes.

In [14]:
def compute_relevance(q, search_function):
    doc_id = q["document"]
    results = search_function(query=q["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["id"] == doc_id))

    return relevance

The total relevance function gets a *search_function* too.

We need to provide it explicitly:

In [15]:
def compute_relevance_total(ground_truth, search_function):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance(q, search_function)
        relevance_total.append(relevance)

    return relevance_total

Use it with *text_search* on the same sample:

In [16]:
relevance_total = compute_relevance_total(ground_truth_sample, text_search)
relevance_total

  0%|          | 0/15 [00:00<?, ?it/s]

[[1, 0, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 1, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [0, 0, 0, 1, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0]]

Now run it for all ground truth questions:

In [17]:
relevance_total = compute_relevance_total(ground_truth, text_search)

  0%|          | 0/565 [00:00<?, ?it/s]

Now we can represent search results as relevance lists. In the next lesson, we'll turn these lists into metrics: Hit Rate and MRR.

## **Search Evaluation Metrics**

In the previous lesson, we computed relevance lists for search results. We can turn those lists into metrics.

## Hit Rate

Hit Rate (also called Recall@k) measures the fraction of queries where the correct document appears anywhere in the results:

In [18]:
example = [
    [1, 0, 0, 0, 0],
    [0, 1, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [0, 0, 0, 0, 0],
    [0, 1, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [0, 0, 1, 0, 0],
    [1, 0, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [1, 0, 0, 0, 0],
]

Each line is one query. If a line contains *1*, search found the correct document somewhere in the top 5 results. If the line contains only zeros, search did not find the correct document.

In our setup, each query has one correct document, so Hit Rate and Recall@k mean the same thing.

Let's calculate it:

In [19]:
cnt = 0

for line in example:
    if 1 in line:
        cnt = cnt + 1

cnt

14

There are 14 hits. The example has 15 queries.

The Hit Rate is:

In [20]:
cnt / len(example)
# 0.933

0.9333333333333333

This means that search found the correct document for 93.3% of the queries in this example.

Put the same logic into a function:

In [21]:
def hit_rate(relevance):
    cnt = 0

    for line in relevance:
        if 1 in line:
            cnt = cnt + 1

    return cnt / len(relevance)

Check it on the same example:

In [22]:
hit_rate(example)
# 0.933

0.9333333333333333

## Mean Reciprocal Rank (MRR)

Hit Rate tells us if we found the right document, but not where it was.

MRR also considers the position.

For each query, the score is based on the rank of the first correct document:

position 1: score is 1.0
position 2: score is 0.5
position 3: score is 0.333
not found: score is 0
In the example, most hits are at the first position. Some hits are lower in the list.

Look at that line:

In [23]:
example[1]
# [0, 1, 0, 0, 0]

[0, 1, 0, 0, 0]

For this line, the score is *1/2* because the correct document is at position 2.

Let's calculate MRR:

In [24]:
total_score = 0.0

for line in example:
    for rank in range(len(line)):
        if line[rank] == 1:
            total_score = total_score + 1 / (rank + 1)
            break

total_score

12.333333333333332

The total score is *12.333333333333334*. We use *rank + 1* because Python counts positions from zero. The first position should score 1/1, and without the *+ 1* we'd divide by zero.

Divide it by the number of queries:

In [25]:
total_score / len(example)
# 0.822

0.8222222222222222

MRR is the average of these scores across all queries. It rewards systems that put the correct document near the top.

Hit Rate is the upper bound for MRR. In practice, MRR is usually smaller because some correct documents are found below the first position.

Put the same logic into a function:

In [26]:
def mrr(relevance):
    total_score = 0.0

    for line in relevance:
        for rank in range(len(line)):
            if line[rank] == 1:
                total_score = total_score + 1 / (rank + 1)
                break

    return total_score / len(relevance)

Check it on the same example:

In [27]:
mrr(example)
# 0.822

0.8222222222222222

# Putting it together

Wrap the metrics in a reusable evaluation function:

In [28]:
def evaluate(ground_truth, search_function):
    relevance_total = compute_relevance_total(ground_truth, search_function)

    return {
        "hit_rate": hit_rate(relevance_total),
        "mrr": mrr(relevance_total),
    }

We can evaluate any search function:

In [29]:
evaluate(
    ground_truth,
    text_search
)

  0%|          | 0/565 [00:00<?, ?it/s]

{'hit_rate': 0.8477876106194691, 'mrr': 0.7358702064896753}

You should see something like:

In [ ]:
{"hit_rate": 0.899, "mrr": 0.769}

Search metrics tell us whether retrieval works. Next, we'll use these metrics to tune the search parameters.

# Interpreting the metrics

A few things to keep in mind when reading these numbers:

Our ground truth assumes only one relevant document per query. In practice, other retrieved documents might also be relevant. A 50% hit rate does not mean that half the results are useless. It means the document we generated the question from did not appear in the top results for half the queries. Other relevant documents may still be there.

With synthetic data, the generated questions can be too close to the original FAQ text. This inflates hit rate and MRR. If you see numbers above 95%, treat them with caution and check whether the questions are realistic enough.

Good thresholds depend on your use case. A 50% hit rate is acceptable for some applications, while others need 90% or higher. The right number depends on how much the downstream LLM can compensate for imperfect retrieval. It also depends on user tolerance for wrong answers.

Look at the system holistically. A high MRR means the relevant document is near the top, which helps the LLM focus on the right context. A low MRR with a high hit rate means the document is there, but buried under less relevant results.

## **Search Parameter Tuning**

In the previous lesson, we defined Hit Rate, MRR, and the *evaluate* function. Now we can use them to tune search parameters.

Instead of guessing which settings are better, we measure them on the ground truth dataset.

So far we've boosted *question* to 3.0. The idea was that a query should match the FAQ question. That kind of match should count for more than matching the answer text. It sounds reasonable. But it's a guess, and now we can check it against data instead of trusting it.

This is the main benefit of offline evaluation. We change one parameter, run the same questions again, and see whether the metric moves. The dataset stays fixed, so the comparison is fair.

## Trying different boosts

Start with a search function where the question boost is configurable:

In [30]:
def search_boost(query, question_boost):
    boost_dict = {"question": question_boost, "section": 0.5}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
    )

Evaluate several boost values:

In [31]:
for boost in [0.5, 1.0, 3.0, 5.0, 10.0]:
    result = evaluate(
        ground_truth,
        lambda query, boost=boost: search_boost(query, boost)
    )
    print(f"boost={boost}: {result}")

  0%|          | 0/565 [00:00<?, ?it/s]

boost=0.5: {'hit_rate': 0.8938053097345132, 'mrr': 0.7912684365781707}


  0%|          | 0/565 [00:00<?, ?it/s]

boost=1.0: {'hit_rate': 0.9061946902654867, 'mrr': 0.7925368731563417}


  0%|          | 0/565 [00:00<?, ?it/s]

boost=3.0: {'hit_rate': 0.8477876106194691, 'mrr': 0.7358702064896753}


  0%|          | 0/565 [00:00<?, ?it/s]

boost=5.0: {'hit_rate': 0.8247787610619469, 'mrr': 0.7040117994100292}


  0%|          | 0/565 [00:00<?, ?it/s]

boost=10.0: {'hit_rate': 0.7982300884955752, 'mrr': 0.6787905604719761}


For the data we prepared on May 29, 2026, this gives:

In [ ]:
boost=0.5: {'hit_rate': 0.9113924050632911, 'mrr': 0.800548523206751}
boost=1.0: {'hit_rate': 0.9240506329113924, 'mrr': 0.8139240506329113}
boost=3.0: {'hit_rate': 0.8987341772151899, 'mrr': 0.7693248945147676}
boost=5.0: {'hit_rate': 0.8708860759493671, 'mrr': 0.7401265822784809}
boost=10.0: {'hit_rate': 0.8582278481012658, 'mrr': 0.7122362869198313}

Increasing the question boost makes the metrics worse, not better. The best value here is *1.0*, no boost at all. That's already the opposite of what the intuition predicted.

But this is only one parameter. We can also tune *answer* and *section* together with *question*.

Define a search function with all three boosts:

In [32]:
def search_boosts(query, question_boost, answer_boost, section_boost):
    boost_dict = {
        "question": question_boost,
        "section": section_boost,
        "answer": answer_boost,
    }

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
    )

Now do a small grid search:

In [33]:
results = []

for question_boost in [1.0, 2.0, 5.0]:
    for answer_boost in [1.0, 2.0, 4.0, 10.0]:
        for section_boost in [0.1, 0.2, 0.5]:
            print(
                f"Evaluating question_boost={question_boost},"
                f" answer_boost={answer_boost},"
                f" section_boost={section_boost}..."
            )
            result = evaluate(
                ground_truth,
                lambda query, question_boost=question_boost, answer_boost=answer_boost, section_boost=section_boost: search_boosts(
                    query,
                    question_boost,
                    answer_boost,
                    section_boost
                )
            )

            results.append({
                "question": question_boost,
                "answer": answer_boost,
                "section": section_boost,
                "hit_rate": result["hit_rate"],
                "mrr": result["mrr"],
            })

Evaluating question_boost=1.0, answer_boost=1.0, section_boost=0.1...


  0%|          | 0/565 [00:00<?, ?it/s]

Evaluating question_boost=1.0, answer_boost=1.0, section_boost=0.2...


  0%|          | 0/565 [00:00<?, ?it/s]

Evaluating question_boost=1.0, answer_boost=1.0, section_boost=0.5...


...

  0%|          | 0/565 [00:00<?, ?it/s]

Evaluating question_boost=5.0, answer_boost=10.0, section_boost=0.2...


  0%|          | 0/565 [00:00<?, ?it/s]

Evaluating question_boost=5.0, answer_boost=10.0, section_boost=0.5...


  0%|          | 0/565 [00:00<?, ?it/s]

Sort by MRR:

In [34]:
df_results = pd.DataFrame(results)
df_results.sort_values("mrr", ascending=False).head(10)

,question,answer,section,hit_rate,mrr
3,1.0,2.0,0.1,0.964602,0.867552
19,2.0,4.0,0.2,0.964602,0.867552
35,5.0,10.0,0.5,0.964602,0.867552
33,5.0,10.0,0.1,0.964602,0.866224
18,2.0,4.0,0.1,0.966372,0.866106
34,5.0,10.0,0.2,0.964602,0.865811
4,1.0,2.0,0.2,0.962832,0.865398
7,1.0,4.0,0.2,0.962832,0.860649
20,2.0,4.0,0.5,0.961062,0.860590
6,1.0,4.0,0.1,0.964602,0.860383


For the same data, the best rows are:

In [ ]:
# question  answer  section  hit_rate  mrr
# 1.0       2.0     0.1      0.975     0.885
# 2.0       4.0     0.2      0.975     0.885
# 5.0       10.0    0.5      0.975     0.885
# 5.0       10.0    0.2      0.975     0.884
# 5.0       10.0    0.1      0.975     0.884
# 2.0       4.0     0.1      0.975     0.884
# 2.0       4.0     0.5      0.977     0.884
# 1.0       2.0     0.2      0.977     0.884
# 1.0       2.0     0.5      0.965     0.862
# 1.0       4.0     0.1      0.970     0.862

The best combination weights *answer* twice as heavily as *question*, with almost no weight on *section*. So the data says the opposite of where we started. The answer text matters more for retrieval than the question text. The intuition was wrong, and we'd never have known without measuring it. This is exactly why we evaluate instead of guess.

The first three rows have the same relative weights:

In [ ]:
# question : answer : section = 1 : 2 : 0.1

So we can use the smaller and easier-to-read values: *question=1.0*, *answer=2.0*, and *section=0.1*. This gives the same relative weights as *question=5.0*, *answer=10.0*, and *section=0.5*, but the numbers are not unnecessarily large.

Define the search function with these boosts:

In [35]:
def text_search(query):
    boost_dict = {
        "question": 1.0,
        "answer": 2.0,
        "section": 0.1,
    }

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
    )

Usually we care about both metrics. Hit Rate tells us whether the correct document appears at all. MRR tells us whether it appears near the top. A document near the top is more likely to be used by the RAG prompt.

## Tuning Workflow

Search parameters can look arbitrary. This includes field boosts, number of results, filters, and other settings. Evaluation gives us a way to compare settings with evidence.

Grid search is fine when there are only a few settings. For a larger parameter space, use a smarter search strategy. You can sample random combinations, use Bayesian optimization, or keep a validation split so you don't overfit the evaluation set.

For text search on our dataset, grid search takes about one second per combination. That makes it practical to try many options. When each evaluation takes minutes instead of seconds, grid search becomes too expensive. In those cases, use Bayesian optimization with a library like hyperopt. It explores the parameter space more efficiently by focusing on combinations that are likely to improve the metric.

## Top-K tradeoffs

We return 5 results from search. Increasing top-K to 10 would improve hit rate because there are more chances to find the correct document. But more results means more context sent to the LLM. That costs more and makes it harder for the model to identify what is relevant. Five results is a reasonable default for short FAQ-style documents.

Next, we'll move from retrieval quality to answer quality and evaluate the full RAG pipeline.